# T1.1 end-to-end smoke: eval-gated retrain on BEIR SciFact

Validates the `vstash.retrain.retrain()` entry point end-to-end using
**real BEIR qrels** (not the internal pseudo-query split). That makes
the NDCG@10 numbers comparable with the published BEIR literature
rather than saturating at 1.0 on diverse corpora.

Steps:
1. Download the full SciFact corpus (5k docs) and ingest it with the
   base model's embeddings.
2. Convert SciFact's labeled queries + qrels into vstash's eval
   format via `qrels_to_eval_queries`.
3. Run `retrain(..., eval_queries=...)` so both baseline and final eval
   use the same labeled set.
4. Report baseline NDCG@10, final NDCG@10, delta, and gate outcome.

Designed to run on Colab (T4 GPU is enough).


In [ ]:
# Cell 1: Setup -- branch feat/retrain-tier1 ships the new retrain() orchestrator
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch feat/retrain-tier1 https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Download SciFact (full corpus) and ingest with base-model embeddings.
# Using the full corpus is what makes the real-qrels eval meaningful: the
# relevant docs must be in the store for the search to find them.
import os
import sys
import shutil

sys.path.insert(0, "/content/vstash")

from experiments.beir_benchmark import download_beir, load_beir
from sentence_transformers import SentenceTransformer
from vstash.store import VstashStore

BASE_MODEL = "BAAI/bge-small-en-v1.5"
STORE_PATH = "/tmp/retrain_t11_scifact.db"
OUTPUT_PATH = "/content/retrained_model"


def path_for_scifact(doc_id):
    return f"scifact://{doc_id}"


# Clean slate
for p in (
    STORE_PATH,
    STORE_PATH + "-wal",
    STORE_PATH + "-shm",
    OUTPUT_PATH,
    OUTPUT_PATH + ".candidate",
):
    if os.path.isdir(p):
        shutil.rmtree(p)
    elif os.path.isfile(p):
        os.remove(p)

cache = download_beir("scifact")
corpus, queries, qrels = load_beir(cache)
doc_ids = list(corpus.keys())
print(f"corpus: {len(doc_ids)} docs  |  queries: {len(queries)}  |  qrels: {len(qrels)}")

# Embed the full corpus with the base model, ingest into vstash.
model = SentenceTransformer(BASE_MODEL)
texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in doc_ids]
vecs = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=128,
)
print(f"embedded {len(texts)} docs, dim={vecs.shape[1]}")

store = VstashStore(STORE_PATH, embedding_dim=int(vecs.shape[1]))
for doc_id, text, vec in zip(doc_ids, texts, vecs):
    store.add_document(
        path=path_for_scifact(doc_id),
        title=corpus[doc_id].get("title", "")[:80] or doc_id,
        chunks=[text],
        embeddings=[list(map(float, vec))],
    )

stats = store.stats()
print(f"vstash store: {stats.documents} docs, {stats.chunks} chunks")

In [ ]:
# Cell 3: Run retrain() with real BEIR qrels as the honest eval set.
import time
from vstash.retrain import qrels_to_eval_queries, retrain as run_retrain

eval_queries = qrels_to_eval_queries(
    queries=queries,
    qrels=qrels,
    path_for_doc_id=path_for_scifact,
)
print(f"eval_queries (real qrels): {len(eval_queries)}")
# Sanity-check: each labeled query should have at least one relevant doc.
n_rel = sum(len(q["relevant_paths"]) for q in eval_queries)
print(f"avg relevant per query: {n_rel / max(len(eval_queries), 1):.2f}")

# Honest eval retrieves over the FULL ingested corpus (~5k docs), not just the
# relevant subset. eval_noise_size caps how many non-relevant chunks we pull
# into the temp eval index, so we set it >= len(corpus) to pull everything.
EVAL_NOISE = max(len(doc_ids), 10000)

t0 = time.perf_counter()
result = run_retrain(
    store,
    base_model=BASE_MODEL,
    output_path=OUTPUT_PATH,
    max_queries=2000,  # training pseudo-queries still come from the corpus
    epochs=2,
    lr=3e-6,
    batch_size=64,
    eval_queries=eval_queries,
    eval_noise_size=EVAL_NOISE,
    min_gain=0.0,
)
elapsed = time.perf_counter() - t0
print(f"\nretrain() finished in {elapsed:.1f}s")
print(f"RetrainResult: {result}")

In [ ]:
# Cell 4: Pretty-printed report
import json
from pathlib import Path


def fmt_metrics(m):
    if m is None:
        return "    (not computed)"
    return (
        f"    n_queries:  {m.n_queries}\n"
        f"    NDCG@10:    {m.ndcg_at_10:.4f}\n"
        f"    MRR:        {m.mrr:.4f}\n"
        f"    Hit@10:     {m.hit_at_10:.4f}"
    )


print("=" * 60)
print("T1.1 end-to-end smoke: SciFact 1k subset")
print("=" * 60)
print(f"n_pairs:       {result.n_pairs}")
print(f"gated_out:     {result.gated_out}")
print(f"min_gain:      {result.min_gain:+.4f}")
print(f"output_path:   {result.output_path}")
print()
print("Baseline:")
print(fmt_metrics(result.baseline))
print()
print("Final:")
print(fmt_metrics(result.final))
print()
if result.baseline is not None and result.final is not None:
    delta = result.delta_ndcg
    print(f"Delta NDCG@10: {delta:+.4f}  ({delta * 100:+.2f}%)")

# Surface training_meta.json so we can eyeball the numbers persisted to disk.
meta_path = Path(result.output_path or (OUTPUT_PATH + ".candidate")) / "training_meta.json"
if meta_path.exists():
    print()
    print("training_meta.json:")
    print(json.dumps(json.loads(meta_path.read_text()), indent=2))